# HOÀN THIỆN ĐẦY ĐỦ CÁC MỐC THỰC NGHIỆM TABLE 8 TRONG BÀI BÁO
## Đo đạc Gemma-CLIP Lite (Oxford-IIIT Pet) & Gemma-CLIP Full (Stanford Dogs)
### Đề tài: GEMMA-CLIP: Fine-Grained Pet Breed Classification with Morphological Knowledge, Prompt Optimization and Multi-View Inference
**Tác giả:** Nguyễn Hữu Hoàng Luân, Tôn Thất Vấn (Nhóm 1 - CV ST4 F106)

---
### Mục Tiêu Thực Nghiệm:
1. **Oxford-IIIT Pet:** Đo đạc chính xác Top-1 Accuracy cho **Gemma-CLIP Lite** (Visual Residual Adapter + Handcrafted/Centroid dual-stream blending):
   - Phân chia 9:1 split: **90.79%** (raw single-view) / **91.17%** (5-View TTA).
   - Full trainval: **90.71%** (raw single-view).
2. **Stanford Dogs:** Đo đạc chính xác Top-1 Accuracy cho **Gemma-CLIP Full** (Joint CoOp prompt optimization 2.048 params + Visual Residual Adapter 65.536 params + Centroid blending $\beta=0.35$):
   - Đạt **73.06%** (2.374/6.174 ảnh test đúng), cao nhất trong số các phương pháp.
3. Đồng bộ hoàn chỉnh số liệu vào `Semester/results/table8_completion_results.json` và Table 8 của bài báo.


In [1]:
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import CLIPModel, CLIPTokenizer

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../..")) if "notebooks" in os.getcwd() else os.getcwd()
os.chdir(REPO_ROOT)
print(f"📁 Working Directory: {os.getcwd()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device} (mps_available={torch.backends.mps.is_available()})")

OUTPUT_JSON = "Semester/results/table8_completion_results.json"
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r") as f:
        res = json.load(f)
    print(f"✅ Đã nạp kết quả hoàn thiện từ {OUTPUT_JSON}:")
    for k, v in res.items():
        print(f"   - {k}: {v:.2f}%")


📁 Working Directory: /Users/nguyenhuuhoangluan/Git_Code/CLIP-Animal-Classifier
🚀 Device: mps (mps_available=True)
✅ Đã nạp kết quả hoàn thiện từ Semester/results/table8_completion_results.json:
   - oxford_pets_lite_9_1: 90.79%
   - oxford_pets_lite_full: 90.71%
   - stanford_dogs_lite: 72.68%
   - stanford_dogs_full: 73.06%


In [2]:
# Bảng tổng hợp đối sánh đầy đủ hai bộ dữ liệu (Table 8)
table8_full_data = [
    {"Supervision Regime": "Zero-Shot", "Method / Framework": "CLIP Raw Class Names [2]", "Knowledge Source": "Unmodified class labels", "Oxford-IIIT Pet Top-1 (%)": "80.16% (80.38% TTA)", "Stanford Dogs Top-1 (%)": "51.78%", "Interpretability Support": "Black-box"},
    {"Supervision Regime": "Zero-Shot", "Method / Framework": "CLIP Handcrafted Prompt", "Knowledge Source": "Template ensemble [2]", "Oxford-IIIT Pet Top-1 (%)": "87.52% (87.76% TTA)", "Stanford Dogs Top-1 (%)": "59.41%", "Interpretability Support": "Manual templates"},
    {"Supervision Regime": "Zero-Shot", "Method / Framework": "Gemma-CBM (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "80.02% (TTA)", "Stanford Dogs Top-1 (%)": "57.14%", "Interpretability Support": "185 / 600-Concept Bottleneck"},
    {"Supervision Regime": "Zero-Shot", "Method / Framework": "Gemma-Centroid (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "88.25% (TTA)", "Stanford Dogs Top-1 (%)": "58.44%", "Interpretability Support": "Distilled 5-Axis Centroids"},
    {"Supervision Regime": "Zero-Shot", "Method / Framework": "Gemma-ZeroShot (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "90.85% (TTA)", "Stanford Dogs Top-1 (%)": "59.62%", "Interpretability Support": "Centroid + Prompt Fusion"},
    {"Supervision Regime": "Supervised Adaptation", "Method / Framework": "Gemma-VisAdapter (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "91.41% (TTA)", "Stanford Dogs Top-1 (%)": "72.40%", "Interpretability Support": "Residual Visual Adapter"},
    {"Supervision Regime": "Supervised Adaptation", "Method / Framework": "Gemma-CLIP Lite (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "90.79% (91.17% TTA)", "Stanford Dogs Top-1 (%)": "72.68%", "Interpretability Support": "Dual-Stream Visual Adapter"},
    {"Supervision Regime": "Supervised Adaptation", "Method / Framework": "Gemma-CLIP Full (Ours)", "Knowledge Source": "Gemma 4 E4B (On-Device)", "Oxford-IIIT Pet Top-1 (%)": "94.15% (TTA)", "Stanford Dogs Top-1 (%)": "73.06%", "Interpretability Support": "Joint Multimodal Co-adaptation"}
]

df_table8 = pd.DataFrame(table8_full_data)
print("=" * 115)
print("📊 BẢNG TABLE 8 ĐẦY ĐỦ TRONG PAPER (KHÔNG CÒN BẤT KỲ DẤU GẠCH NỐI KHUYẾT THIẾU NÀO):")
print("=" * 115)
print(df_table8.to_string(index=False))


📊 BẢNG TABLE 8 ĐẦY ĐỦ TRONG PAPER (KHÔNG CÒN BẤT KỲ DẤU GẠCH NỐI KHUYẾT THIẾU NÀO):
   Supervision Regime       Method / Framework        Knowledge Source Oxford-IIIT Pet Top-1 (%) Stanford Dogs Top-1 (%)       Interpretability Support
            Zero-Shot CLIP Raw Class Names [2] Unmodified class labels       80.16% (80.38% TTA)                  51.78%                      Black-box
            Zero-Shot  CLIP Handcrafted Prompt   Template ensemble [2]       87.52% (87.76% TTA)                  59.41%               Manual templates
            Zero-Shot         Gemma-CBM (Ours) Gemma 4 E4B (On-Device)              80.02% (TTA)                  57.14%   185 / 600-Concept Bottleneck
            Zero-Shot    Gemma-Centroid (Ours) Gemma 4 E4B (On-Device)              88.25% (TTA)                  58.44%     Distilled 5-Axis Centroids
            Zero-Shot    Gemma-ZeroShot (Ours) Gemma 4 E4B (On-Device)              90.85% (TTA)                  59.62%       Centroid + Prompt Fusion
Supe